In [5]:
#Bloco 1 Importar bibliotecas e carregar dados
import pandas as pd
import numpy as np

# Carrega os dados necessários
params_df    = pd.read_csv('../outputs/params_table.csv')
features_df  = pd.read_csv('../outputs/clustering_table.csv')

# Carrega a BoM (tabela DiBA)
xl = pd.ExcelFile('../data/Industrial_case_20251203.xlsx')
print("Abas disponíveis:", xl.sheet_names)
bom = pd.read_excel('../data/Industrial_case_20251203.xlsx', 
                     sheet_name='4. DiBA')

print(f"Shape: {bom.shape}")
print(bom.head(10))
print(f"\nColunas: {bom.columns.tolist()}")

Abas disponíveis: ['1. OrdiniCliente', '2. OrdiniProduzione', 'Foglio1', '3. SchemaProduzione', '4. DiBA', 'Pivot Demand']
Shape: (12135, 5)
  Production BOM No_  Version Code  Line No_  Type      No_
0             300001           NaN     10000     1  S300001
1             300002           NaN     10000     1  S300002
2             300003           NaN     10000     1  S300780
3             300004           NaN     10000     1  S300003
4             300005           NaN     10000     1  S300039
5             300006           NaN     10000     1  S300918
6             300007           NaN     10000     1  S300004
7             300008           NaN     10000     1  S300784
8             300009           NaN     10000     1  S300005
9             300010           NaN     10000     1  S300006

Colunas: ['Production BOM No_', 'Version Code', 'Line No_', 'Type', 'No_']


In [19]:
# ── DECISÃO DE DESIGN: Cascata Simplificada (v1) ──
#
# A cascata completa requer uma BoM com quantidades por nível
# para propagar a demanda de produtos acabados para barras
# extrudadas e billets. A tabela DiBA disponível contém apenas
# a relação pai-filho entre itens, sem quantidades de conversão.
#
# Adicionalmente, o OrdiniProduzione não é uma fonte confiável
# para definir o nível de estoque ideal — ele reflete decisões
# históricas da empresa, não o que o clustering recomenda.
#
# Por isso, na v1 do módulo, o nível de produção é determinado
# diretamente pelo policy_cluster, sem propagação via BoM.
# Essa é uma simplificação consciente e documentada.
# A cascata completa pode ser implementada numa v2 quando
# a BoM com quantidades estiver disponível.
#
# Regras adotadas:
# MTS           → finished_products  (estoque de produto acabado)
# Comakership   → finished_products  (estoque planejado com cliente)
# Batch-to-Order→ finished_products  (produz em lote sob pedido)
# Dynamic       → finished_products  (política dinâmica no nível FP)
# MTO           → raw_material       (sem estoque, só matéria-prima)

def assign_level(cluster):
    if cluster == 'MTO':
        return 'raw_material'
    else:
        return 'finished_products'

final_df = params_df.copy()
final_df['production_level'] = final_df['policy_cluster'].apply(assign_level)

print("Distribuição por nível de produção:")
print(final_df['production_level'].value_counts())

print("\nDistribuição por cluster:")
print(final_df['policy_cluster'].value_counts())

print("\nVerifica itens conhecidos:")
itens_check = ['300091', '300007', '300092', '300019']
check = final_df[final_df['Item'].astype(str).isin(itens_check)][
    ['Item', 'policy_cluster', 'production_level', 'SS', 'ROP', 'EOQ']
]
print(check.to_string(index=False))

Distribuição por nível de produção:
production_level
finished_products    966
raw_material         422
Name: count, dtype: int64

Distribuição por cluster:
policy_cluster
Dynamic        721
MTO            422
MTS            241
Comakership      4
Name: count, dtype: int64

Verifica itens conhecidos:
  Item policy_cluster  production_level      SS     ROP     EOQ
300007            MTO      raw_material    0.00    0.00    0.00
300019        Dynamic finished_products 3182.28 3328.96 1405.58
300091            MTS finished_products  550.06  750.03 1641.16
300092        Dynamic finished_products 1312.85 1378.87  942.95


In [22]:
#Bloco 3 Salvar
final_df.to_csv('../outputs/cascade_table.csv', index=False)
print("Cascade table salva com sucesso.")
print(f"\nResumo final:")
print(final_df.groupby(['production_level', 'policy_cluster']).size().to_string())

Cascade table salva com sucesso.

Resumo final:
production_level   policy_cluster
finished_products  Comakership         4
                   Dynamic           721
                   MTS               241
raw_material       MTO               422
